# Comptage de mots : Python simple vs pseudo-MapReduce vs Hadoop MapReduce

## 0. Introduction et objectifs

Nous comparons trois implémentations du comptage d'occurrences de mots :
1. Python séquentiel (dictionnaire) ;
2. Pseudo-MapReduce en Python (map → shuffle → reduce en local) ;
3. Hadoop MapReduce (pseudo-distribué, HDFS + YARN).

**Hypothèses à tester :**
    
- H1 : sur petits fichiers, Python est plus rapide car Hadoop paie un coût fixe de démarrage.
- H2 : le temps de Hadoop croît moins vite avec la taille, d'où un éventuel point de bascule.
- H3 : la version pseudo-MapReduce consomme plus de mémoire (stockage de toutes les paires).

In [1]:
# Configuration du projet : dossier de travail fixé explicitement
import os
from pathlib import Path

PROJET = Path.home() / "tp_wordcount"
os.chdir(PROJET)
for dossier in ["data", "src", "results"]:
    (PROJET / dossier).mkdir(exist_ok=True)

print("Dossier de travail :", os.getcwd())
print("Contenu :", sorted(os.listdir()))

Dossier de travail : /home/fadwa_hdd/tp_wordcount
Contenu : ['.ipynb_checkpoints', '.venv', 'data', 'results', 'src', 'wordcount_comparaison.ipynb']


In [2]:
import platform, sys, subprocess, psutil

def cmd(c):
    """Exécute une commande shell et renvoie la première ligne de sa sortie."""
    try:
        r = subprocess.run(c, shell=True, capture_output=True, text=True)
        out = (r.stdout or r.stderr).strip()
        return out.splitlines()[0] if out else "non trouvé"
    except Exception as e:
        return f"indisponible ({e})"

print("OS        :", platform.platform())
print("Python    :", sys.version.split()[0])
print("CPU       :", cmd("grep -m1 'model name' /proc/cpuinfo").split(":")[-1].strip())
print("Cœurs     :", psutil.cpu_count(logical=False), "physiques /", psutil.cpu_count(), "logiques")
print("RAM       :", round(psutil.virtual_memory().total / 1e9, 1), "Go")
print("Disque    :", round(psutil.disk_usage('/').free / 1e9, 1), "Go libres")
print("Java      :", cmd("java -version"))
print("Hadoop    :", cmd("hadoop version"))

OS        : Linux-6.18.40.1-microsoft-standard-WSL2-x86_64-with-glibc2.35
Python    : 3.10.12
CPU       : 12th Gen Intel(R) Core(TM) i5-1235U
Cœurs     : 6 physiques / 12 logiques
RAM       : 8.2 Go
Disque    : 1019.2 Go libres
Java      : openjdk version "11.0.32.1" 2026-08-18
Hadoop    : /bin/sh: 1: hadoop: not found


In [3]:
print("Espace libre réel (C:) :", round(psutil.disk_usage('/mnt/c').free / 1e9, 1), "Go")

Espace libre réel (C:) : 79.5 Go


**Remarques sur l'environnement :**
- Tout s'exécute dans WSL2 (Ubuntu 22.04), sur le système de fichiers Linux, pour éviter le coût d'accès à `/mnt/c`.
- Python 3.10 (le document utilise 3.11 : sans impact ici). Java 11, version validée pour Hadoop 3.4.
- CPU hybride de portable (i5-1235U) : mesures faites sur secteur, en mode performances, et répétées pour lisser la variabilité.
- 8,2 Go de RAM alloués à WSL : c'est cette limite qui borne la taille des fichiers testés.

## 2. Génération des données

- Vocabulaire : les 15 mots du document de référence, tirés selon une distribution de type Zipf (poids 1/rang), plus réaliste qu'un tirage uniforme.
- Graine aléatoire fixée pour la reproductibilité.
- Fichier de base : N = 100 000 lignes, un mot par ligne.
- Fichiers dérivés : le fichier de base dupliqué m fois, pour m ∈ {1, 5, 10, 50, 100, 200, 500}.

La duplication donne une **vérité terrain exacte** : compte attendu = compte dans le fichier de base × m. Chaque méthode sera validée contre cette référence avant toute mesure de temps.

In [4]:
import random
from pathlib import Path
from collections import Counter

DATA = Path("data")
DATA.mkdir(exist_ok=True)

VOCAB = ["chat", "chien", "arbre", "fleur", "maison", "soleil", "montagne", "livre",
         "ciel", "mer", "lune", "route", "oiseau", "jardin", "voiture"]
N = 100_000
SEED = 42

# Distribution de type Zipf : le mot de rang r a un poids 1/r
poids = [1 / (r + 1) for r in range(len(VOCAB))]
rng = random.Random(SEED)
mots = rng.choices(VOCAB, weights=poids, k=N)

base = DATA / f"base_N{N}.txt"
base.write_text("\n".join(mots) + "\n", encoding="utf-8")

ref_base = Counter(mots)   # vérité terrain pour m = 1
print(f"{base} : {N} lignes, {base.stat().st_size / 1e6:.2f} Mo")
print("Les 5 mots les plus fréquents :", ref_base.most_common(5))

data/base_N100000.txt : 100000 lignes, 0.59 Mo
Les 5 mots les plus fréquents : [('chat', 30137), ('chien', 15038), ('arbre', 10000), ('fleur', 7455), ('maison', 6103)]


In [5]:
import time
import pandas as pd

M_VALUES = [1, 5, 10, 50, 100, 200, 500]
contenu = base.read_bytes()   # environ 0,6 Mo, lu une seule fois

fichiers = {}
lignes_tab = []
for m in M_VALUES:
    f = DATA / f"mots_m{m}.txt"
    t0 = time.perf_counter()
    if not f.exists():                 # ne régénère pas si le fichier existe déjà
        with open(f, "wb") as out:
            for _ in range(m):         # écriture en flux : pas de copie géante en mémoire
                out.write(contenu)
    fichiers[m] = f
    lignes_tab.append({"m": m,
                       "lignes": N * m,
                       "taille (Mo)": round(f.stat().st_size / 1e6, 1),
                       "génération (s)": round(time.perf_counter() - t0, 2)})

df_fichiers = pd.DataFrame(lignes_tab)
df_fichiers

,m,lignes,taille (Mo),génération (s)
0,1,100000,0.6,0.0
1,5,500000,3.0,0.0
2,10,1000000,5.9,0.0
3,50,5000000,29.5,0.0
4,100,10000000,59.1,0.0
5,200,20000000,118.1,0.0
6,500,50000000,295.4,0.0


In [6]:
def reference(m):
    """Résultat attendu pour le fichier dupliqué m fois."""
    return {mot: n * m for mot, n in ref_base.items()}

def verifier(resultat, m):
    """Vérifie qu'un résultat est exactement égal à la vérité terrain."""
    attendu = reference(m)
    if dict(resultat) != attendu:
        diff = {k: (resultat.get(k), attendu.get(k))
                for k in set(resultat) | set(attendu)
                if resultat.get(k) != attendu.get(k)}
        raise AssertionError(f"Résultat incorrect pour m={m} : {diff}")
    return True

# Contrôle indépendant avec wc -l (outil système, pas notre code)
import subprocess
for m in [1, 500]:
    n_lignes = int(subprocess.run(["wc", "-l", str(fichiers[m])],
                                  capture_output=True, text=True).stdout.split()[0])
    assert n_lignes == N * m, f"m={m} : {n_lignes} lignes au lieu de {N * m}"
print("Nombre de lignes vérifié avec wc -l ✔")

Nombre de lignes vérifié avec wc -l ✔


## 3. Version 1 : Python simple

### Protocole de mesure (commun aux trois versions)
- Chaque exécution est lancée dans un processus séparé, comme en ligne de commande. Le temps mesuré inclut donc le démarrage de l'interpréteur, de même que le temps Hadoop inclut celui de la JVM.
- Une exécution d'échauffement valide le résultat contre la vérité terrain et met le fichier en cache disque. Toutes les méthodes sont ainsi mesurées à cache chaud.
- 5 répétitions : on retient la médiane (et on note le min et le max).
- Pic mémoire mesuré avec `/usr/bin/time` (RSS maximal).
- Limite mémoire de 6,5 Go pour les scripts Python, pour que le dépassement donne une `MemoryError` propre plutôt qu'un blocage de WSL.

### Implémentation
Code identique au document. Seule modification : le nom du fichier est passé en argument au lieu d'être écrit en dur.

In [7]:
%%writefile src/wordcount_v1.py
"""Version 1 : comptage séquentiel avec un dictionnaire (fidèle au document)."""
import sys

def compter_occurrences(fichier):
    with open(fichier, 'r', encoding='utf-8') as f:
        mots = f.read().split()
    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1
    return compteur

if __name__ == "__main__":
    fichier = sys.argv[1] if len(sys.argv) > 1 else "mots.txt"
    resultats = compter_occurrences(fichier)
    for mot, nb in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {nb}")

Overwriting src/wordcount_v1.py


In [8]:
import subprocess, sys, time, resource, statistics
from pathlib import Path

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
LIMITE_MEMOIRE = int(6.5e9)   # octets

def _limiter_memoire():
    """Appliquée au processus enfant : au-delà de la limite, Python lève MemoryError."""
    resource.setrlimit(resource.RLIMIT_AS, (LIMITE_MEMOIRE, LIMITE_MEMOIRE))

def executer(cmd, limiter=True):
    """Lance cmd dans un processus séparé.
    Renvoie (durée en s, pic mémoire en Mo, stdout, code retour, stderr)."""
    t0 = time.perf_counter()
    r = subprocess.run(["/usr/bin/time", "-f", "%M"] + cmd,
                       capture_output=True, text=True,
                       preexec_fn=_limiter_memoire if limiter else None)
    duree = time.perf_counter() - t0
    try:
        pic_mo = int(r.stderr.strip().splitlines()[-1]) / 1024   # %M est en Ko
    except (ValueError, IndexError):
        pic_mo = float("nan")
    return duree, pic_mo, r.stdout, r.returncode, r.stderr

def parser_sortie(stdout):
    """Convertit les lignes 'mot : nb' en dictionnaire."""
    resultat = {}
    for ligne in stdout.strip().splitlines():
        mot, nb = ligne.rsplit(":", 1)
        resultat[mot.strip()] = int(nb)
    return resultat

def benchmark(nom, cmd_pour, m_values, repetitions=5):
    """Échauffement et validation, puis `repetitions` mesures, pour chaque m."""
    lignes = []
    for m in m_values:
        cmd = cmd_pour(fichiers[m])

        # 1) Échauffement et validation du résultat
        duree, pic, out, rc, err = executer(cmd)
        if rc != 0:
            statut = "échec mémoire" if "MemoryError" in err else f"échec (code {rc})"
            print(f"[{nom}] m={m:>4} : {statut}")
            lignes.append({"methode": nom, "m": m, "lignes": N * m, "statut": statut})
            break   # inutile de tester des fichiers plus gros
        verifier(parser_sortie(out), m)

        # 2) Mesures
        temps, pics = [], []
        for _ in range(repetitions):
            duree, pic, out, rc, err = executer(cmd)
            temps.append(duree)
            pics.append(pic)

        lignes.append({"methode": nom, "m": m, "lignes": N * m,
                       "taille_Mo": round(fichiers[m].stat().st_size / 1e6, 1),
                       "temps_median_s": round(statistics.median(temps), 4),
                       "temps_min_s": round(min(temps), 4),
                       "temps_max_s": round(max(temps), 4),
                       "memoire_pic_Mo": round(max(pics), 1),
                       "statut": "ok"})
        print(f"[{nom}] m={m:>4} : {statistics.median(temps):.3f} s (médiane), "
              f"{max(pics):.0f} Mo, résultat validé ✔")

    df = pd.DataFrame(lignes)
    df.to_csv(RESULTS / f"{nom}.csv", index=False)
    return df

In [9]:
temps_demarrage = statistics.median(
    executer([sys.executable, "-c", "pass"])[0] for _ in range(10)
)
print(f"Démarrage de l'interpréteur Python seul : {temps_demarrage * 1000:.1f} ms")

Démarrage de l'interpréteur Python seul : 15.2 ms


In [10]:
df_v1 = benchmark("v1_python_simple",
                  lambda f: [sys.executable, "src/wordcount_v1.py", str(f)],
                  M_VALUES)
df_v1

[v1_python_simple] m=   1 : 0.028 s (médiane), 16 Mo, résultat validé ✔
[v1_python_simple] m=   5 : 0.073 s (médiane), 45 Mo, résultat validé ✔
[v1_python_simple] m=  10 : 0.130 s (médiane), 83 Mo, résultat validé ✔
[v1_python_simple] m=  50 : 0.638 s (médiane), 435 Mo, résultat validé ✔
[v1_python_simple] m= 100 : 1.583 s (médiane), 754 Mo, résultat validé ✔
[v1_python_simple] m= 200 : 3.063 s (médiane), 1499 Mo, résultat validé ✔
[v1_python_simple] m= 500 : 8.090 s (médiane), 3735 Mo, résultat validé ✔


,methode,m,lignes,taille_Mo,temps_median_s,temps_min_s,temps_max_s,memoire_pic_Mo,statut
0,v1_python_simple,1,100000,0.6,0.0283,0.0281,0.0296,15.6,ok
1,v1_python_simple,5,500000,3.0,0.0730,0.0714,0.0786,45.4,ok
2,v1_python_simple,10,1000000,5.9,0.1297,0.1235,0.1345,82.8,ok
3,v1_python_simple,50,5000000,29.5,0.6375,0.6271,0.6806,434.5,ok
4,v1_python_simple,100,10000000,59.1,1.5828,1.5499,1.6365,753.7,ok
5,v1_python_simple,200,20000000,118.1,3.0633,2.9099,3.2223,1499.0,ok
6,v1_python_simple,500,50000000,295.4,8.0899,7.5534,9.5085,3735.4,ok


### Analyse de la V1
- **Démarrage :** l'interpréteur Python seul coûte environ 15 ms. Sur 100 000 mots, la V1 prend 28 ms, dont environ 13 ms de calcul réel. Le temps de 0,012 s mesuré dans le document correspondait donc surtout au lancement de Python.
- **Temps :** croissance presque linéaire. Environ 130 ms par million de mots jusqu'à 5 M de mots, puis environ 155 à 160 ms au-delà (8,1 s pour 50 M de mots). Le changement de palier coïncide avec le passage à plusieurs centaines de Mo de mémoire.
- **Mémoire :** environ 75 octets par mot (une chaîne Python par mot, plus un pointeur dans la liste, plus le texte brut lu d'un bloc), soit 3,7 Go pour 50 M de mots, conformément à l'estimation.
- **Facteur limitant :** le processeur. Le débit (environ 6 M de mots/s, soit 37 Mo/s) est très inférieur à la vitesse de lecture du cache.
- **Limite de passage à l'échelle :** `f.read().split()` charge tout le fichier en mémoire. Avec 8 Go de RAM, la V1 ne peut pas traiter plus d'environ 100 M de mots.

## 4. Version 2 : pseudo-MapReduce Python

Reproduction locale des trois phases du modèle :
- **map** : chaque ligne produit des paires `(mot, 1)` ;
- **shuffle** : les paires sont regroupées par clé (`mot → [1, 1, ...]`) ;
- **reduce** : les valeurs de chaque clé sont sommées.

Code identique au document. Seule modification : le nom du fichier est passé en argument.

**Prévision avant mesure :** la lecture se fait ligne par ligne, mais toutes les paires sont accumulées dans `mapped` avant le shuffle. Coût estimé : environ 140 octets par mot (chaîne, tuple, pointeur, puis entier dans la liste du shuffle), soit près de deux fois la V1. On prévoit donc un dépassement de la limite de 6,5 Go pour m = 500 (environ 7 Go) et un temps supérieur à celui de la V1.

In [11]:
%%writefile src/wordcount_v2.py
"""Version 2 : pseudo-MapReduce local (fidèle au document)."""
import sys
from collections import defaultdict

def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced

if __name__ == "__main__":
    fichier = sys.argv[1] if len(sys.argv) > 1 else "mots.txt"
    mapped = []
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            mapped.extend(mapper(ligne))

    grouped = shuffle(mapped)
    reduced = reducer(grouped)

    for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {count}")

Writing src/wordcount_v2.py


In [12]:
df_v2 = benchmark("v2_pseudo_mapreduce",
                  lambda f: [sys.executable, "src/wordcount_v2.py", str(f)],
                  M_VALUES)
df_v2

[v2_pseudo_mapreduce] m=   1 : 0.059 s (médiane), 23 Mo, résultat validé ✔
[v2_pseudo_mapreduce] m=   5 : 0.253 s (médiane), 78 Mo, résultat validé ✔
[v2_pseudo_mapreduce] m=  10 : 0.523 s (médiane), 147 Mo, résultat validé ✔
[v2_pseudo_mapreduce] m=  50 : 2.655 s (médiane), 698 Mo, résultat validé ✔
[v2_pseudo_mapreduce] m= 100 : 5.663 s (médiane), 1387 Mo, résultat validé ✔
[v2_pseudo_mapreduce] m= 200 : 12.144 s (médiane), 2765 Mo, résultat validé ✔
[v2_pseudo_mapreduce] m= 500 : échec mémoire


,methode,m,lignes,taille_Mo,temps_median_s,temps_min_s,temps_max_s,memoire_pic_Mo,statut
0,v2_pseudo_mapreduce,1,100000,0.6,0.0586,0.0562,0.0593,22.8,ok
1,v2_pseudo_mapreduce,5,500000,3.0,0.2529,0.2354,0.2674,78.2,ok
2,v2_pseudo_mapreduce,10,1000000,5.9,0.5234,0.5211,0.5443,147.3,ok
3,v2_pseudo_mapreduce,50,5000000,29.5,2.6549,2.5657,2.6951,698.0,ok
4,v2_pseudo_mapreduce,100,10000000,59.1,5.6628,5.5127,5.9507,1387.3,ok
5,v2_pseudo_mapreduce,200,20000000,118.1,12.1443,11.4966,12.7021,2765.5,ok
6,v2_pseudo_mapreduce,500,50000000,NaN,NaN,NaN,NaN,NaN,échec mémoire


### Analyse de la V2
- **H3 validée :** environ 138 octets par mot (prévision : 140), soit 1,8 fois la V1. Pour m = 500 (50 M de mots), le besoin estimé d'environ 7 Go dépasse la limite : `MemoryError`, comme prévu.
- **Temps :** environ 4 fois plus lent que la V1 dès que le calcul domine le démarrage. Causes : un appel de fonction par ligne, un tuple créé par mot, et deux passes supplémentaires (shuffle, puis reduce).
- **Correction du document :** la « surcharge faible » observée sur un petit fichier n'était qu'un effet du temps de démarrage. À l'échelle, la surcharge est d'un facteur 4.
- **Limite structurelle :** la V2 reproduit la *forme* du modèle MapReduce, mais pas ses *mécanismes de passage à l'échelle*. Toutes les paires intermédiaires restent en mémoire, alors que Hadoop les écrit sur disque (*spill*), les trie, et les réduit localement grâce à un *combiner* avant le shuffle.

## 5. Version 3 : Hadoop MapReduce

### Installation (réalisée dans le terminal WSL)
- **Hadoop 3.4.3** au lieu de 3.4.0 : même branche 3.4, même modèle MapReduce et même exemple WordCount (corrections de bugs uniquement). La 3.4.0 n'est plus servie que par le serveur d'archives d'Apache, très lent. La 3.4.3 est distribuée par le CDN et n'embarque pas le SDK AWS (environ 500 Mo inutiles ici).
- Intégrité de l'archive vérifiée par somme SHA-512.
- Java 11 (`JAVA_HOME` déclaré aussi dans `hadoop-env.sh`, car les scripts de démarrage passent par SSH).
- SSH sans mot de passe vers `localhost`, nécessaire aux scripts `start-dfs.sh` et `start-yarn.sh`. Correction préalable d'une option invalide dans `/etc/ssh/ssh_config`.
- **À chaque redémarrage de WSL :** relancer `sudo service ssh start` avant de démarrer Hadoop.

### Configuration pseudo-distribuée (écrite par le notebook)
- `dfs.replication = 1` (un seul nœud).
- Données HDFS dans `~/hadoop_data` plutôt que `/tmp`, qui est effacé à chaque redémarrage de WSL.
- YARN limité à 4 Go et conteneurs de 1 Go, adaptés aux 8,2 Go de RAM de WSL.
- Mémoire maximale des services Hadoop limitée à 512 Mo chacun.
- Contrôle de la mémoire virtuelle désactivé (faux positifs fréquents avec Java 11 sous WSL).

In [14]:
import os, subprocess
from pathlib import Path

JAVA_HOME   = "/usr/lib/jvm/java-11-openjdk-amd64"
HADOOP_HOME = Path.home() / "hadoop"
HADOOP_CONF = HADOOP_HOME / "etc" / "hadoop"
HADOOP_DATA = Path.home() / "hadoop_data"

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
os.environ["HADOOP_CONF_DIR"] = str(HADOOP_CONF)
os.environ["PDSH_RCMD_TYPE"] = "ssh"
chemins = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:{JAVA_HOME}/bin"
if chemins not in os.environ["PATH"]:
    os.environ["PATH"] = chemins + ":" + os.environ["PATH"]

print(subprocess.run(["hadoop", "version"], capture_output=True, text=True).stdout.splitlines()[0])

Hadoop 3.4.3


In [15]:
import os, subprocess
from pathlib import Path

JAVA_HOME   = "/usr/lib/jvm/java-11-openjdk-amd64"
HADOOP_HOME = Path.home() / "hadoop"
HADOOP_CONF = HADOOP_HOME / "etc" / "hadoop"
HADOOP_DATA = Path.home() / "hadoop_data"

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
os.environ["HADOOP_CONF_DIR"] = str(HADOOP_CONF)
os.environ["PDSH_RCMD_TYPE"] = "ssh"
chemins = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:{JAVA_HOME}/bin"
if chemins not in os.environ["PATH"]:
    os.environ["PATH"] = chemins + ":" + os.environ["PATH"]

print(subprocess.run(["hadoop", "version"], capture_output=True, text=True).stdout.splitlines()[0])

Hadoop 3.4.3


In [16]:
import shutil

def ecrire_xml(nom, proprietes):
    """Écrit un fichier de configuration Hadoop. L'original est sauvegardé une seule fois en .orig."""
    chemin = HADOOP_CONF / nom
    sauvegarde = chemin.with_suffix(".xml.orig")
    if chemin.exists() and not sauvegarde.exists():
        shutil.copy(chemin, sauvegarde)
    lignes = ['<?xml version="1.0" encoding="UTF-8"?>', "<configuration>"]
    for cle, valeur in proprietes.items():
        lignes += ["  <property>", f"    <name>{cle}</name>",
                   f"    <value>{valeur}</value>", "  </property>"]
    lignes.append("</configuration>")
    chemin.write_text("\n".join(lignes) + "\n")
    print(f"✔ {nom}")

ecrire_xml("core-site.xml", {
    "fs.defaultFS": "hdfs://localhost:9000",
    "hadoop.tmp.dir": f"{HADOOP_DATA}/tmp",
})

ecrire_xml("hdfs-site.xml", {
    "dfs.replication": "1",
    "dfs.namenode.name.dir": f"file://{HADOOP_DATA}/namenode",
    "dfs.datanode.data.dir": f"file://{HADOOP_DATA}/datanode",
})

ecrire_xml("mapred-site.xml", {
    "mapreduce.framework.name": "yarn",
    "mapreduce.application.classpath":
        "$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/*:$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/lib/*",
    "yarn.app.mapreduce.am.env": f"HADOOP_MAPRED_HOME={HADOOP_HOME}",
    "mapreduce.map.env":         f"HADOOP_MAPRED_HOME={HADOOP_HOME}",
    "mapreduce.reduce.env":      f"HADOOP_MAPRED_HOME={HADOOP_HOME}",
    "yarn.app.mapreduce.am.resource.mb": "1024",
    "mapreduce.map.memory.mb": "1024",
    "mapreduce.reduce.memory.mb": "1024",
})

ecrire_xml("yarn-site.xml", {
    "yarn.nodemanager.aux-services": "mapreduce_shuffle",
    "yarn.nodemanager.env-whitelist":
        "JAVA_HOME,HADOOP_COMMON_HOME,HADOOP_HDFS_HOME,HADOOP_CONF_DIR,"
        "CLASSPATH_PREPEND_DISTCACHE,HADOOP_YARN_HOME,HADOOP_HOME,PATH,LANG,TZ,HADOOP_MAPRED_HOME",
    "yarn.nodemanager.resource.memory-mb": "4096",
    "yarn.scheduler.minimum-allocation-mb": "512",
    "yarn.scheduler.maximum-allocation-mb": "4096",
    "yarn.nodemanager.vmem-check-enabled": "false",
})

# Mémoire maximale des services Hadoop (lue par hadoop-env.sh, car les services démarrent via SSH)
env_sh = HADOOP_CONF / "hadoop-env.sh"
contenu_env = env_sh.read_text()
for ligne in [f"export JAVA_HOME={JAVA_HOME}", "export HADOOP_HEAPSIZE_MAX=512m"]:
    if ligne not in contenu_env:
        with open(env_sh, "a") as f:
            f.write(ligne + "\n")
print("✔ hadoop-env.sh")

✔ core-site.xml
✔ hdfs-site.xml
✔ mapred-site.xml
✔ yarn-site.xml
✔ hadoop-env.sh


In [17]:
if not (HADOOP_DATA / "namenode" / "current").exists():
    r = subprocess.run(["hdfs", "namenode", "-format", "-nonInteractive"],
                       capture_output=True, text=True)
    print("\n".join((r.stdout + r.stderr).splitlines()[-5:]))
else:
    print("HDFS déjà formaté : formatage ignoré (il effacerait les données).")

2026-10-13 11:36:56,493 INFO namenode.FSImage: FSImageSaver clean checkpoint: txid=0 when meet shutdown.
2026-10-13 11:36:56,494 INFO namenode.NameNode: SHUTDOWN_MSG: 
/************************************************************
SHUTDOWN_MSG: Shutting down NameNode at FadwaUniv/127.0.1.1
************************************************************/


In [18]:
import time

for script in ["start-dfs.sh", "start-yarn.sh"]:
    r = subprocess.run([script], capture_output=True, text=True)
    print(f"$ {script}\n{r.stdout}{r.stderr}")

time.sleep(5)   # laisse le temps aux services Java de démarrer
print(subprocess.run(["jps"], capture_output=True, text=True).stdout)

$ start-dfs.sh
Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [FadwaUniv]
FadwaUniv: Warning: Permanently added 'fadwauniv' (ED25519) to the list of known hosts.

$ start-yarn.sh
Starting resourcemanager
Starting nodemanagers

21088 DataNode
21648 NodeManager
22020 Jps
21318 SecondaryNameNode
21528 ResourceManager
20955 NameNode

